In [1]:
!pip install --no-cache-dir transformers sentencepiece

In [2]:
!pip install pandas

In [3]:
!pip install protobuf

In [4]:
!pip install tqdm

In [5]:
!pip install sacrebleu

In [6]:
!pip install joblib

# Evaluation

In [7]:
import os
import pandas as pd

all_translations = {}
refs = {}
src = {}

for model_name in ["mbart", "j2", "gpt", "deepseek"]:
    for translation_name in ["news", "poetry", "literature"]:
        file_prefix = model_name+"_"+translation_name
        print("Reading file: " + file_prefix, end=". ")
        data = pd.read_csv(os.path.join("new_data", file_prefix +".out.csv"), sep='\t', header=0)
        _tmp_lines = data['translation'].tolist()
        all_translations[file_prefix] = _tmp_lines
        print(str(len(_tmp_lines)) + " read.")

for translation_name in ["news", "poetry", "literature"]:
    file_prefix = translation_name + "-1000.csv"
    print("Reading file: " + file_prefix, end=". ")
    if translation_name in ["literature"]:
        true = pd.read_csv(file_prefix, sep=';', header=0, encoding='utf8', engine='python')
    else:
        true = pd.read_csv(file_prefix, sep=';', header=0, encoding='CP1252', engine='python')

    refs[translation_name] = true['NL'].tolist()
    src[translation_name] = true['EN'].tolist()
    print(str(len(refs[translation_name])) + " read.")
    print(str(len(src[translation_name])) + " read.")


Reading file: mbart_news. 1012 read.
Reading file: mbart_poetry. 1000 read.
Reading file: mbart_literature. 1000 read.
Reading file: j2_news. 1012 read.
Reading file: j2_poetry. 1000 read.
Reading file: j2_literature. 1000 read.
Reading file: gpt_news. 1012 read.
Reading file: gpt_poetry. 1000 read.
Reading file: gpt_literature. 1000 read.
Reading file: deepseek_news. 1012 read.
Reading file: deepseek_poetry. 1000 read.
Reading file: deepseek_literature. 1000 read.
Reading file: news-1000.csv. 1012 read.
1012 read.
Reading file: poetry-1000.csv. 1000 read.
1000 read.
Reading file: literature-1000.csv. 1000 read.
1000 read.


In [3]:
from sacrebleu.metrics import BLEU, CHRF, TER

for translation_name in ["news", "poetry", "literature"]:
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name

        print(exp_prefix)
        print(BLEU().corpus_score(all_translations[exp_prefix], [refs[translation_name]], n_bootstrap = 1000), 
                CHRF().corpus_score(all_translations[exp_prefix], [refs[translation_name]], n_bootstrap = 1000),
                TER().corpus_score(all_translations[exp_prefix], [refs[translation_name]], n_bootstrap = 1000)
             )

mbart_news
BLEU = 34.81 (μ = 34.82 ± 1.28) 67.1/41.7/28.2/19.7 (BP = 0.986 ratio = 0.986 hyp_len = 20144 ref_len = 20421) chrF2 = 62.19 (μ = 62.19 ± 0.88) TER = 50.57 (μ = 50.57 ± 1.36)
j2_news
BLEU = 38.38 (μ = 38.39 ± 1.24) 68.8/45.2/32.1/23.4 (BP = 0.982 ratio = 0.982 hyp_len = 20052 ref_len = 20421) chrF2 = 65.60 (μ = 65.61 ± 0.89) TER = 48.50 (μ = 48.50 ± 1.32)
gpt_news
BLEU = 39.79 (μ = 39.79 ± 1.29) 69.9/46.4/33.4/24.6 (BP = 0.985 ratio = 0.985 hyp_len = 20122 ref_len = 20421) chrF2 = 66.72 (μ = 66.73 ± 0.90) TER = 47.19 (μ = 47.19 ± 1.39)
deepseek_news
BLEU = 19.12 (μ = 19.16 ± 2.20) 44.1/23.6/14.3/9.0 (BP = 1.000 ratio = 1.379 hyp_len = 28155 ref_len = 20421) chrF2 = 54.45 (μ = 54.46 ± 1.58) TER = 97.00 (μ = 97.12 ± 13.34)
mbart_poetry
BLEU = 20.63 (μ = 20.64 ± 1.56) 45.9/25.4/15.7/9.9 (BP = 1.000 ratio = 1.119 hyp_len = 8269 ref_len = 7391) chrF2 = 46.39 (μ = 46.40 ± 1.44) TER = 69.41 (μ = 69.35 ± 2.05)
j2_poetry
BLEU = 27.67 (μ = 27.72 ± 1.66) 52.6/32.6/22.2/15.4 (BP = 1.000

In [8]:
from sacrebleu.metrics import BLEU, CHRF, TER
from joblib import Parallel, delayed
from scipy.stats import ttest_ind

def get_bleu(sys, ref):
    ''' Computing BLEU using sacrebleu

        :param sysname: the name of the system
        :param sys: the sampled sentences from the translation (type = list)
        :param ref: the reference sentences (type = list)
        :returns: a socre (float)
    '''
    bleu = BLEU().corpus_score(sys, [ref])
    return bleu.score

def get_chrf(sys, ref):
    ''' Computing CHRF using sacrebleu

        :param sysname: the name of the system
        :param sys: the sampled sentences from the translation (type = list)
        :param ref: the reference sentences (type = list)
        :returns: a socre (float)
    '''
    chrf = CHRF().corpus_score(sys, [ref])
    return chrf.score

def get_ter(sys, ref):
    ''' Computing BLEU using sacrebleu

        :param sysname: the name of the system
        :param sys: the sampled sentences from the translation (type = list)
        :param ref: the reference sentences (type = list)
        :returns: a socre (float)
    '''
    ter = TER().corpus_score(sys, [ref])
    return ter.score

def compute_metric(metric_func, sentences, ref, sample_idxs, iters):
    ''' Computing metric

        :param metric_func: get_bleu or get_ter_multeval
        :param sys: the sampled sentences from the translation
        :param ref: the reference sentences
        :param lang: the langauge for detokenization
        :param sample_idxs: indexes for the sample (list)
        :param iters: number of iterations
        :returns: a socre (float)
    '''
    # 5. let's get the measurements for each sample
    scores = {}
    scores = Parallel(n_jobs=8)(delayed(eval(metric_func))([sentences[j] for j in sample_idxs[i]], [ref[j] for j in sample_idxs[i]]) for i in range(iters))
             
    return scores
    
    
def compute_significance(metrics, iterations):
    ''' Compute pairwose significance interval

        :param metrics: dictionary with systems and metrics
        :param iterations: the number of iterations
        :returns: a socre (float)
    '''
    # now, we are able to compute statistical significance
    # print('delta(xi) > delta(x):')
    scores = {}
    for system1 in metrics:
        scores[system1] = {}
        for system2 in metrics:
            s = 0.0
            for i in range(iterations):
                if round(metrics[system1][i], 4) > round(metrics[system2][i], 4):
                    s += 1.0
            if system1 == system2:
                scores[system1][system2] = -1.0
            else:
                scores[system1][system2] = s / iterations
    return scores
    

def compute_ttest_scikit(metrics, iterations):
    ''' Compute pairwose significance interval

        :param metrics: dictionary with systems and metrics
        :param iterations: the number of iterations
        :returns: a socre (float)
    '''
    scores = {}
    print('\nScikit ttest:')
    for system1 in metrics:
        scores[system1] = {}
        for system2 in metrics:
            t, p =  ttest_ind(metrics[system1], metrics[system2])
            scores[system1][system2] = p
        
    return scores


def print_latex_table(scores, metric_title):
    ''' Prints a table in latex format; ready to incorporate into a tex file

        :param scores: dictionary with scores and systems
        :param metric_title: identifying the metric (string)
    '''
    print(' '.join([str(s) for s in scores.keys()]))
    for system in scores:
        print(system + ' ' + ' '.join([str(p) for p in scores[system].values()]))

    print('\n')
    print('  & ' + ' & '.join([str(s) for s in scores[system].keys()]) + '\\\\\hline')
    for system in scores:
        print_en = False
        print(system, end='')
        for system2 in scores:
            p = scores[system][system2]
            if (p >= 0.0 and p < 0.05) or p >= 0.95:
                if print_en:
                    print(' & Y', end='')
                else:
                    print(' & ', end='')
            else:
                if print_en:
                    print(' & N', end='')
                else:
                    print(' & ', end='')

            if system == system2:
                print_en = True
        print('\\\\\hline')


# 3. read the other variables.
import numpy as np

iters = int(1000)
sample_size = int(300)

# 4. Compute Sample metric
metrics = {}

metrics['bleu'] = {}
metrics['chrf'] = {}
metrics['ter'] = {}
    
for translation_name in ["news", "poetry", "literature"]:
    for metric in metrics:
        for model_name in ["mbart", "j2", "gpt", "deepseek"]:
            exp_prefix = model_name+"_"+translation_name
            #print(exp_prefix)
            sample_idxs = np.random.randint(0, len(refs[translation_name]), size=(iters, sample_size))
            metrics[metric][model_name] = compute_metric('get_'+metric, all_translations[exp_prefix], refs[translation_name], sample_idxs, iters)
            
        print("-------------------------------------------------")
        print(translation_name + " " + metric)
        sign_scores = compute_significance(metrics[metric], iters)
        print_latex_table(sign_scores, metric)
        sign_scores = compute_ttest_scikit(metrics[metric], iters)
        print_latex_table(sign_scores, metric)

-------------------------------------------------
news bleu
mbart j2 gpt deepseek
mbart -1.0 0.013 0.004 1.0
j2 0.987 -1.0 0.222 1.0
gpt 0.996 0.778 -1.0 1.0
deepseek 0.0 0.0 0.0 -1.0


  & mbart & j2 & gpt & deepseek\\\hline
mbart &  & Y & Y & Y\\\hline
j2 &  &  & N & Y\\\hline
gpt &  &  &  & Y\\\hline
deepseek &  &  &  & \\\hline

Scikit ttest:
mbart j2 gpt deepseek
mbart 1.0 0.0 0.0 0.0
j2 0.0 1.0 2.804489151903412e-124 0.0
gpt 0.0 2.804489151903412e-124 1.0 0.0
deepseek 0.0 0.0 0.0 1.0


  & mbart & j2 & gpt & deepseek\\\hline
mbart &  & Y & Y & Y\\\hline
j2 &  &  & Y & Y\\\hline
gpt &  &  &  & Y\\\hline
deepseek &  &  &  & \\\hline
-------------------------------------------------
news chrf
mbart j2 gpt deepseek
mbart -1.0 0.005 0.0 1.0
j2 0.995 -1.0 0.159 1.0
gpt 1.0 0.841 -1.0 1.0
deepseek 0.0 0.0 0.0 -1.0


  & mbart & j2 & gpt & deepseek\\\hline
mbart &  & Y & Y & Y\\\hline
j2 &  &  & N & Y\\\hline
gpt &  &  &  & Y\\\hline
deepseek &  &  &  & \\\hline

Scikit ttest:
mbart j2 g

In [16]:
!pip install unbabel-comet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 45.0 MB/s eta 0:00:001m49.8 MB/s eta 0:00:01
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Installing backend dependencies ... one
done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.9 MB/s eta 0:00:00MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 42.3 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 44.8 MB/s eta 0:00:00m eta 0:00:0

In [22]:
from comet import download_model

model_names = ["wmt22-comet-da"]
model_paths = {}
for model_name in model_names:
    
    model_path = download_model("/".join(["Unbabel", model_name]))
    model_paths[model_name] = model_path

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [25]:
from comet import load_from_checkpoint

comet_scores = {}
model_names = ["wmt22-comet-da"]
for model_name in model_names:
    model = load_from_checkpoint(model_paths[model_name])

    data = []
    for translation_name in ["news", "poetry", "literature"]:
        for trans_model_name in ["mbart", "j2", "gpt", "deepseek"]:
            exp_prefix = trans_model_name+"_"+translation_name
            data = [{"src": src, "mt": mt, "ref": ref} for (src, mt, ref) in zip(src[translation_name], all_translations[exp_prefix], refs[translation_name])]
            comet_scores[model_name + exp_prefix] = model.predict(data, batch_size=8)
            

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Predicting DataLoader 0:  10%|█▎            | 12/125 [20:08<3:09:38, 100.69s/it]
Encoder model frozen.
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|████████████████| 127/127 [06:13<00:00,  2.94s

In [28]:
model_names = ["wmt22-comet-da"]
for model_name in model_names:
    for translation_name in ["news", "poetry", "literature"]:
        for trans_model_name in ["mbart", "j2", "gpt", "deepseek"]:
            exp_prefix = trans_model_name+"_"+translation_name
            print(model_name + " " + exp_prefix + ": " + str(comet_scores[model_name + exp_prefix]['system_score']))

wmt22-comet-da mbart_news: 0.8903198004240104
wmt22-comet-da j2_news: 0.9011142428802408
wmt22-comet-da gpt_news: 0.9070505630004077
wmt22-comet-da deepseek_news: 0.8554555743697132
wmt22-comet-da mbart_poetry: 0.7082525874078274
wmt22-comet-da j2_poetry: 0.740972532838583
wmt22-comet-da gpt_poetry: 0.7577246414721012
wmt22-comet-da deepseek_poetry: 0.6877447870373726
wmt22-comet-da mbart_literature: 0.8149589537978172
wmt22-comet-da j2_literature: 0.8518830806910992
wmt22-comet-da gpt_literature: 0.8590605648756027
wmt22-comet-da deepseek_literature: 0.7844721899330616


In [30]:
!pip install --upgrade pip  # ensures that pip is current
!git clone https://github.com/google-research/bleurt.git
!cd bleurt && pip install .

fatal: destination path 'bleurt' already exists and is not an empty directory.
Processing /home/dimitarsh1/Documents/LLMs/LLM-MT-Eval-Noa/bleurt
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.8/620.8 MB 41.1 MB/s  0:00:14 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 34.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 34.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 39.7 MB/s  0:00:00 eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 37.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 42.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 40.9 MB/s  0:00:00 44.3 MB/s eta 0:00:01
  Created wheel for BLEURT: filename=bleurt-0.0.2-py3-none-any.wh

In [3]:
from bleurt import score

checkpoint = "bleurt/BLEURT-20-D3"
scorer = score.BleurtScorer(checkpoint)

bluert_scores = {}
for translation_name in ["news", "poetry", "literature"]:
        for trans_model_name in ["mbart", "j2", "gpt", "deepseek"]:
            exp_prefix = trans_model_name+"_"+translation_name
            candidates = all_translations[exp_prefix]
            references = refs[translation_name]
            bluert_scores[checkpoint + exp_prefix] = scorer.score(references = references, candidates = candidates)

INFO:tensorflow:Reading checkpoint bleurt/BLEURT-20-D3.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint BLEURT-20-D3
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:BLEURT-20-D3
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... max_seq_length:512
INFO:tensorflow:... vocab_file:None
INFO:tensorflow:... do_lower_case:None
INFO:tensorflow:... sp_model:sent_piece
INFO:tensorflow:... dynamic_seq_length:True
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Will load model: bleurt/BLEURT-20-D3/sent_piece.model.
INFO:tensorflow:SentencePiece tokenizer created.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.


2025-12-09 16:12:56.703916: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


NameError: name 'all_translations' is not defined

In [2]:
from bleurt import score

checkpoint = "bleurt/BLEURT-20-D6"
scorer = score.BleurtScorer(checkpoint)

bluert_scores = {}
for translation_name in ["news", "poetry", "literature"]:
        for trans_model_name in ["mbart", "j2", "gpt", "deepseek"]:
            exp_prefix = trans_model_name+"_"+translation_name
            candidates = all_translations[exp_prefix]
            references = refs[translation_name]
            bluert_scores[checkpoint + exp_prefix] = scorer.score(references = references, candidates = candidates)

2025-12-09 16:12:26.411634: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 16:12:26.443317: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-09 16:12:27.210358: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


INFO:tensorflow:Reading checkpoint bleurt/BLEURT-20-D6.


AssertionError: Could not find BLEURT checkpoint bleurt/BLEURT-20-D6

In [1]:
for score in bluert_scores:
    print(bluert_scores[score])

NameError: name 'bluert_scores' is not defined

In [11]:
def mean(values: list[float]) -> float:
    return float(sum(values) / len(values)) if values else 0.0

for score in bluert_scores:
    print(score, end=": ")
    print (mean(bluert_scores[score]))

bleurt/BLEURT-20-D3mbart_news: 0.6080530961246594
bleurt/BLEURT-20-D3j2_news: 0.617155446247621
bleurt/BLEURT-20-D3gpt_news: 0.6321222839265944
bleurt/BLEURT-20-D3deepseek_news: 0.5579000728226933
bleurt/BLEURT-20-D3mbart_poetry: 0.49143901717662813
bleurt/BLEURT-20-D3j2_poetry: 0.5303624999970198
bleurt/BLEURT-20-D3gpt_poetry: 0.5568385994881392
bleurt/BLEURT-20-D3deepseek_poetry: 0.47621953776478765
bleurt/BLEURT-20-D3mbart_literature: 0.5139609934389591
bleurt/BLEURT-20-D3j2_literature: 0.5568444950133562
bleurt/BLEURT-20-D3gpt_literature: 0.5722993874847889
bleurt/BLEURT-20-D3deepseek_literature: 0.4845194829553366


In [16]:
!pip install lexical_diversity
!pip install lexicalrichness
!pip install spacy_udpipe

In [17]:
import itertools
from lexical_diversity import lex_div as ld
from lexicalrichness import LexicalRichness as lr
from scipy.stats import ttest_ind
from joblib import Parallel, delayed
import statistics
import spacy_udpipe
import time
import pickle
import os
from nltk.probability import FreqDist
import logging

def plot_freqdist_freq(fd,
                       max_num=None,
                       cumulative=False,
                       title='Frequency plot',
                       linewidth=2):
    """
    As of NLTK version 3.2.1, FreqDist.plot() plots the counts
    and has no kwarg for normalising to frequency.
    Work this around here.

    INPUT:
        - the FreqDist object
        - max_num: if specified, only plot up to this number of items
          (they are already sorted descending by the FreqDist)
        - cumulative: bool (defaults to False)
        - title: the title to give the plot
        - linewidth: the width of line to use (defaults to 2)
    OUTPUT: plot the freq and return None.
    """

    tmp = fd.copy()
    norm = fd.N()
    for key in tmp.keys():
        tmp[key] = float(fd[key]) / norm

    if max_num:
        tmp.plot(max_num, cumulative=cumulative,
                 title=title, linewidth=linewidth)
    else:
        tmp.plot(cumulative=cumulative,
                 title=title,
                 linewidth=linewidth)

    return

def get_lemmas(sentences, nlpD, system_name, freq_voc = None, cache = True):
    ''' Computes the lemmas and their frequencies for the given sentences

        :param sentences: a list of sentences
        :param nlpd: the data model for the lematizer
        :param freq_voc: a frequency vocabulary
        :param cache: whether to create new file or not (True).
        :returns: a dictionary of lemmas and frequencies
    '''
    a = time.time()

    lemmas = {}

    if os.path.exists(system_name + ".spacy_udpipe.lemmas") and cache:
        logging.debug("Lemmas dict loading from file")
        with open(system_name + ".spacy_udpipe.lemmas", "rb") as SpUpM:
            lemmas = pickle.load(SpUpM)
        logging.debug("Lemmas dict loaded")
    else:
        logging.debug("Lemmas dict building from scratch")
        #nlps = list(nlpD.pipe(sentences, n_process=-1))
        nlps = list(nlpD.pipe(sentences))

        for doc in nlps:
            for token in doc:
                lemma=token.lemma_
                tokenLow=str(token).lower()

                if lemma in lemmas: # existing lemma
                    if tokenLow not in lemmas[lemma]:
                        lemmas[lemma][tokenLow]=1
                    else:
                        lemmas[lemma][tokenLow]+=1
                else:                       # unexisting lemma
                    lemmas[lemma]={}        # if this is the first time we have a lemma then there are no tokens
                    lemmas[lemma][tokenLow]=1

        with open(system_name + ".spacy_udpipe.lemmas", "wb") as PoF:
            pickle.dump(lemmas, PoF)

        logging.debug("Lemmas dict built and saved")

    #print("Length of all lemmas: " + str(len(lemmas)))
    all_lemas_len = len(lemmas)
    singleton_lemmas = [lemma + "\t" + str(len(lemmas[lemma])) for lemma in lemmas if len(lemmas[lemma]) < 2]
    #print("Length of singleton lemmas: " + str(len(singleton_lemmas)))
    all_sing_lemmas_len = len(singleton_lemmas)
    singleton_matching_lemmas = []

    with open(system_name + ".lemmas", "w") as oF:
        oF.write("\n".join([lemma + ": " + "\t".join(str(f) + "|" + str(g) for (f,g) in zip(lemmas[lemma].keys(), lemmas[lemma].values())) for lemma in lemmas]))

    if freq_voc is not None:
        tmp_lemmas = {}
        for lemma in lemmas:
            if len(lemmas[lemma]) > 1:
                for form in lemmas[lemma]:
                    if form in freq_voc:
                        tmp_lemmas[lemma] = lemmas[lemma]
                        break           # we only need one occurance to match
            else:
                singleton_matching_lemmas.append(lemma)
        lemmas = tmp_lemmas

    # print("Length of matched lemmas: " + str(len(lemmas)))
    # print("Length of singleton maching lemmas: " + str(len(singleton_matching_lemmas)))

    return (lemmas, singleton_lemmas)

def simpson_diversity(wordFormDict):
    ''' Computes the Simpson Diversity Index

        :param wordFormDict: a dictionary { 'wordform': count }
        :returns: diversity index (number)
    '''

    def p(n, N):
        ''' Relative abundance
        '''
        if n ==  0:
            return 0
        else:
            return float(n)/N

    N = sum(wordFormDict.values())
    return sum(p(n, N)**2 for n in wordFormDict.values() if n != 0)

def inverse_simpson_diversity(wordFormDict):
    ''' Computes the inverse Simpson Diversity Index
    
        :param wordFormDict: a dictionary { 'wordform': count }
        :returns: diversity index (number) 
    '''
    return float(1)/simpson_diversity(wordFormDict)

"""# Shannon Diversity #
The Shannon-Weiner diversity represent the proportion of species abundance in the population. Its being at maximum when all species occur in similar number of individuals and the lowest when the sample contain one species. From my experience there is no limit to compare the diversity value with as for evenness, which resricted between 0-1. For Example, if the sample contain 4 species each represented by 5o individuals the, diversity H equal 1.3863, and if the sample contain 5 species (one more) and each represented by similar number of individuals (50), the diversity equal 1.6094.
"""

def shannon_diversity(wordFormDict):
    '''
    
        :param wordFormDict: a dictionary { 'species': count }
        :returns: Shannon Diversity Index
    '''
    #>>> sdi({'a': 10, 'b': 20, 'c': 30,})
    #1.0114042647073518
    
    from math import log as ln
    
    def p(n, N):
        """ Relative abundance """
        if n ==  0:
            return 0
        else:
            return (float(n)/N) * ln(float(n)/N)
            
    N = sum(wordFormDict.values())
    
    return -sum(p(n, N) for n in wordFormDict.values() if n != 0)

def compute_simpDiv(nestedDict):
    ''' Computes the simpson diversity for every lemma
        example input : {lemma1:{wordf1: count1, wordf2: count2}, lemma2 {wordform1: count1}}
        output {lemma1: simpDiv1, lemma2:simpDiv2}
        
        :param nestedDict: a nested dictionary
        :returns: a dictionary with the simpson diversity for every lemma 
    '''
    simpsonDict = {}
    for l in nestedDict:
        simpsonDict[l]=simpson_diversity(nestedDict[l])
    return statistics.mean(simpsonDict.values())

def compute_invSimpDiv(nestedDict):
    ''' Computes the simpson diversity for every lemma
        example input : {lemma1:{wordf1: count1, wordf2: count2}, lemma2 {wordform1: count1}}
        output {lemma1: simpDiv1, lemma2:simpDiv2}
    
        :param nestedDict: a dictionary of dictionaries
        :returns: a dictionary with the inversed simpson diversity
    '''
    simpsonDict={}
    for l in nestedDict:
        simpsonDict[l]=inverse_simpson_diversity(nestedDict[l])
    return statistics.mean(simpsonDict.values()) 

def compute_shannonDiv(nestedDict):
    ''' Computes the shannon diversity for every lemma
        example input : {lemma1:{wordf1: count1, wordf2: count2}, lemma2 {wordform1: count1}}
        output {lemma1: simpDiv1, lemma2:simpDiv2}
        
        :param nestedDict: a dictionary of dictionaries
        :returns: a dictionary with the shannon diversity
    '''
    shannonDict={}
    for lem in nestedDict:
        shannonDict[lem]=shannon_diversity(nestedDict[lem])
    return statistics.mean(shannonDict.values())

def compute_yules_i(sentences):
    ''' Computing Yules I measure

        :param sentences: dictionary with all words and their frequencies
        :returns: Yules I (the inverse of yule's K measure) (float) - the higher the better
    '''
    _total, vocabulary = get_vocabulary(sentences)
    M1 = float(len(vocabulary))
    M2 = sum([len(list(g))*(freq**2) for freq,g in itertools.groupby(sorted(vocabulary.values()))])

    try:
        return (M1*M1)/(M2-M1)
    except ZeroDivisionError:
        return 0

def compute_ttr(sentences):
    ''' Computes the type token ratio
    
        :param sentences: the sentences
        :returns: The type token ratio (float)
    '''      

    total, vocabulary = get_vocabulary(sentences)    
    return len(vocabulary)/total
    
def compute_mtld(sentences):
    ''' Computes the MTLD
    
        :param sentences: sentences
    
        :returns: The MTLD (float)
    '''      
    
    def my_mtld(lex, threshold, reverse=False):
        """
        Parameters
        ----------
        threshold: float
            Factor threshold for MTLD. Algorithm skips to a new segment when TTR goes below the
            threshold (default=0.72).
        reverse: bool
            If True, compute mtld for the reversed sequence of text (default=False).
        Returns:
            mtld measure (float)
        """
        if reverse:
            word_iterator = iter(reversed(lex.wordlist))
        else:
            word_iterator = iter(lex.wordlist)

        terms = set()
        word_counter = 0
        factor_count = 0

        for word in word_iterator:
            word_counter += 1
            terms.add(word)
            ttr = len(terms)/word_counter

            if ttr <= threshold:
                word_counter = 0
                terms = set()
                factor_count += 1

        # partial factors for the last segment computed as the ratio of how far away ttr is from
        # unit, to how far away threshold is to unit
        if word_counter > 0:
            factor_count += (1-ttr) / (1 - threshold)

        # ttr never drops below threshold by end of text
        if factor_count == 0:
            ttr = lex.terms / lex.words
            if ttr == 1:
                factor_count += 1
            else:
                factor_count += (1-ttr) / (1 - threshold)

        return len(lex.wordlist) / factor_count

    ll = '\n'.join(sentences)
    lex = lr(ll)
    return lex.mtld()
#    return ld.mtld(ll)
    
def get_vocabulary(sentence_array):
    ''' Compute vocabulary

        :param sentence_array: a list of sentences
        :returns: a list of tokens
    '''
    data_vocabulary = {}
    total = 0
    
    for sentence in sentence_array:
        for token in sentence.strip().split():
            if token not in data_vocabulary:
                data_vocabulary[token] = 1 #/len(line.strip().split())
            else:
                data_vocabulary[token] += 1 #/len(line.strip().split())
            total += 1
            
    return total, data_vocabulary

def get_vocabulary_lowercase(sentence_array):
    ''' Compute vocabulary but converts everything to lowercase first

        :param sentence_array: a list of sentences
        :returns: a list of tokens
    '''
    data_vocabulary = {}
    total = 0
    
    for sentence in sentence_array:
        for token in sentence.lower().strip().split():
            if token.lower() not in data_vocabulary:
                data_vocabulary[token.lower()] = 1 #/len(line.strip().split())
            else:
                data_vocabulary[token.lower()] += 1 #/len(line.strip().split())
            total += 1
            
    return total, data_vocabulary

def compute_ld_metric(metric_func, sentences, sample_idxs, iters):
    ''' Computing metric

        :param metric_func: get_bleu or get_ter_multeval
        :param sys: the sampled sentences from the translation
        :param sample_idxs: indexes for the sample (list)
        :param iters: number of iterations
        :returns: a socre (float)
    '''
    # 5. let's get the measurements for each sample
    scores = Parallel(n_jobs=-1)(delayed(eval(metric_func))([sentences[j] for j in sample_idxs[i]]) for i in range(iters))

    return scores

def compute_gram_diversity(sentences, lang="en", system_name="", freq_voc=None, cache=True):
    ''' Computing metric

        :param metric_func: get_bleu or get_ter_multeval
        :param sys: the sampled sentences from the translation
        :param sample_idxs: indexes for the sample (list)
        :param iters: number of iterations
        :returns: a socre (float)
    '''
    nlpD = spacy_udpipe.load(lang).tokenizer
    nlpD.max_length = 300000000

    (lemmas, singleton_lemmas) = get_lemmas(sentences, nlpD, system_name, freq_voc, cache)

    return (len(lemmas), len(singleton_lemmas), compute_simpDiv(lemmas), compute_invSimpDiv(lemmas), compute_shannonDiv(lemmas))

def textToLFP(sentences, step=1000, last=2000):
    '''we are not lowercasing, tokenizing, removing stopwords, numerals etc.
    this is because we are looking into algorithmic bias and as such into the effect of the algorithm
    on the text it is offered. The text is already tokenized. Might add Lowercasing too.'''

    #create Frequency Dictionary
    fdist = FreqDist(" ".join(sentences).split()) # our text is already tokenized. We merge all sentences together
                                                  # and create one huge list of tokens.

    # get size range
    end = last + step
    sizes = list(range(0, end, step))

    #Get words for every frequency band
    freqs = [fdist.most_common(size+step)[size:size+step] for size in sizes[:-1]]
    freqs.append(fdist.most_common()[last:])

    #total tokens
    totalCount=fdist.N()

    #percentage frequency band
    percs = [sum([count for (_word,count) in freq])/totalCount for freq in freqs]

    #plot
    #plot_freqdist_freq(fdist, 20)

    return percs

In [7]:
!pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 7.8 MB/s  0:00:00


In [10]:
from sacremoses import MosesTokenizer
mtok = MosesTokenizer(lang='nl')

all_tok_translations = {}
refs_tok = {}

for translation_name in ["news", "poetry", "literature"]:
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        all_tok_translations[exp_prefix] = [" ".join(mtok.tokenize(_tok_sent, escape=False)) for _tok_sent in all_translations[exp_prefix]]

    refs_tok[translation_name] = [" ".join(mtok.tokenize(_tok_sent, escape=False)) for _tok_sent in refs[translation_name]]

In [28]:
# ----------------------------------
# Yules I, TTR, MTLD (non tokenized)
# ----------------------------------
print("Domain & model & Yule's I & TTR & MTLD\\\\\hline")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end="")
    print(" & Ref.", end=" & ")
    lex_scores = [compute_yules_i(refs[translation_name]),
              compute_ttr(refs[translation_name]),
              compute_mtld(refs[translation_name])]
    print(" & ".join([str(round(score, 4)) for score in lex_scores]) + "\\\\\\hline")
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        lex_scores = [compute_yules_i(all_translations[exp_prefix]),
              compute_ttr(all_translations[exp_prefix]),
              compute_mtld(all_translations[exp_prefix])]
        print(" & ".join([str(round(score, 4)) for score in lex_scores]) + "\\\\\\hline")
    

Domain & model & Yule's I & TTR & MTLD\\\hline
news & Ref. & 12.5203 & 0.3139 & 103.2077\\\hline
 & mbart & 10.8292 & 0.2953 & 97.1528\\\hline
 & j2 & 11.5953 & 0.3025 & 103.8332\\\hline
 & gpt & 11.1829 & 0.2988 & 101.0221\\\hline
 & deepseek & 12.3972 & 0.2839 & 125.358\\\hline
poetry & Ref. & 29.993 & 0.4269 & 179.4295\\\hline
 & mbart & 26.803 & 0.3945 & 123.8495\\\hline
 & j2 & 27.3999 & 0.4129 & 153.651\\\hline
 & gpt & 25.1865 & 0.4033 & 159.8977\\\hline
 & deepseek & 23.3805 & 0.31 & 177.8315\\\hline
literature & Ref. & 12.4881 & 0.2914 & 131.4198\\\hline
 & mbart & 11.547 & 0.2901 & 117.9076\\\hline
 & j2 & 12.7824 & 0.2967 & 126.5212\\\hline
 & gpt & 12.5634 & 0.2943 & 127.9486\\\hline
 & deepseek & 16.8334 & 0.2872 & 163.3832\\\hline


In [29]:
# ------------------------------
# Yules I, TTR, MTLD (TOKENIZED)
# ------------------------------
print("Domain & model & Yule's I & TTR & MTLD\\\\\hline")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end="")
    print(" & Ref", end=" & ")
    lex_scores = [compute_yules_i(refs_tok[translation_name]),
              compute_ttr(refs_tok[translation_name]),
              compute_mtld(refs_tok[translation_name])]
    print(" & ".join([str(round(score, 4)) for score in lex_scores]) + "\\\\\\hline")
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        lex_scores = [compute_yules_i(all_tok_translations[exp_prefix]),
              compute_ttr(all_tok_translations[exp_prefix]),
              compute_mtld(all_tok_translations[exp_prefix])]
        print(" & ".join([str(round(score, 4)) for score in lex_scores]) + "\\\\\\hline")

    
    

Domain & model & Yule's I & TTR & MTLD\\\hline
news & Ref & 5.4398 & 0.2321 & 103.2077\\\hline
 & mbart & 4.2812 & 0.2106 & 97.1528\\\hline
 & j2 & 5.0875 & 0.2266 & 103.915\\\hline
 & gpt & 5.0562 & 0.2248 & 100.7675\\\hline
 & deepseek & 5.396 & 0.2101 & 125.3914\\\hline
poetry & Ref & 12.4715 & 0.3384 & 182.6479\\\hline
 & mbart & 3.8876 & 0.2704 & 122.7931\\\hline
 & j2 & 3.8601 & 0.2882 & 152.2353\\\hline
 & gpt & 7.1788 & 0.3015 & 159.1791\\\hline
 & deepseek & 7.7445 & 0.2237 & 174.4561\\\hline
literature & Ref & 4.1752 & 0.2063 & 131.4198\\\hline
 & mbart & 3.6372 & 0.1966 & 116.9994\\\hline
 & j2 & 4.1934 & 0.2086 & 127.043\\\hline
 & gpt & 4.2391 & 0.2073 & 127.5569\\\hline
 & deepseek & 5.6489 & 0.2046 & 162.9625\\\hline


In [30]:
# -------------------------
# LEXICAL FREQUENCY PROFILE
# -------------------------
print("Lexical Frequency Profile")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end=" & ")
    print("Refs.", end = " & ")
    print(" & ".join([str(round(band, 4)) for band in textToLFP(refs_tok[translation_name])]), end="\\\\\hline\n")
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        print(" & ".join([str(round(band, 4)) for band in textToLFP(all_tok_translations[exp_prefix])]), end="\\\\\hline\n")



Lexical Frequency Profile
news & Refs. & 0.7821 & 0.0833 & 0.1345\\\hline
 & mbart & 0.8056 & 0.0827 & 0.1117\\\hline
 & j2 & 0.7872 & 0.0856 & 0.1272\\\hline
 & gpt & 0.7882 & 0.086 & 0.1257\\\hline
 & deepseek & 0.7652 & 0.0852 & 0.1496\\\hline
poetry & Refs. & 0.7959 & 0.1343 & 0.0697\\\hline
 & mbart & 0.8515 & 0.1219 & 0.0266\\\hline
 & j2 & 0.8327 & 0.1209 & 0.0464\\\hline
 & gpt & 0.8247 & 0.1263 & 0.049\\\hline
 & deepseek & 0.7911 & 0.0834 & 0.1255\\\hline
literature & Refs. & 0.8113 & 0.0766 & 0.1121\\\hline
 & mbart & 0.8229 & 0.0729 & 0.1042\\\hline
 & j2 & 0.8081 & 0.0762 & 0.1157\\\hline
 & gpt & 0.8083 & 0.0763 & 0.1154\\\hline
 & deepseek & 0.7764 & 0.0781 & 0.1456\\\hline


In [22]:
# Downloading the nl model for spacy udpipe
spacy_udpipe.download("nl")
spacy_udpipe.download("en")

Already downloaded a model for the 'nl' language
Already downloaded a model for the 'en' language


In [32]:
# ----------------------------------------------
# GRAM DIVERSITY (Shannon, Inv Simpson, Simpson)
# ----------------------------------------------
print("Domain & model & Num. all lemmas & Num. signle lemmas & Shannon & Inv. Simpson & Simpson\\\\\hline")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end="")
    print(" & Ref.", end=" & ")
    gram_div_scores = compute_gram_diversity(refs_tok[translation_name], "nl", translation_name, None, False)
    print(" & ".join([str(round(score, 4)) for score in gram_div_scores ]) + "\\\\hline")
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        gram_div_scores = compute_gram_diversity(all_tok_translations[exp_prefix], "nl", exp_prefix, None, False)
        print(" & ".join([str(round(score, 4)) for score in gram_div_scores]) + "\\\\hline")

Domain & model & Num. all lemmas & Num. signle lemmas & Shannon & Inv. Simpson & Simpson\\\hline
news & Ref. & 3959 & 3434 & 0.9388 & 1.13 & 0.0943\\hline
 & mbart & 3516 & 3007 & 0.9332 & 1.1426 & 0.1035\\hline
 & j2 & 3744 & 3215 & 0.9347 & 1.1391 & 0.1012\\hline
 & gpt & 3697 & 3152 & 0.9317 & 1.1456 & 0.1053\\hline
 & deepseek & 4930 & 4242 & 0.9363 & 1.1351 & 0.099\\hline
poetry & Ref. & 2138 & 1922 & 0.9497 & 1.1179 & 0.079\\hline
 & mbart & 1814 & 1578 & 0.9363 & 1.1499 & 0.1005\\hline
 & j2 & 1975 & 1752 & 0.9433 & 1.1347 & 0.0897\\hline
 & gpt & 1971 & 1744 & 0.943 & 1.1367 & 0.0903\\hline
 & deepseek & 3915 & 3472 & 0.9471 & 1.1183 & 0.0829\\hline
literature & Ref. & 3578 & 3069 & 0.9314 & 1.1521 & 0.1068\\hline
 & mbart & 3490 & 2971 & 0.9293 & 1.154 & 0.11\\hline
 & j2 & 3714 & 3192 & 0.933 & 1.1498 & 0.1046\\hline
 & gpt & 3675 & 3126 & 0.9288 & 1.1567 & 0.1103\\hline
 & deepseek & 5214 & 4536 & 0.9385 & 1.1351 & 0.0956\\hline


In [24]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stops_nl = stopwords.words('dutch')


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dimitarsh1/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [31]:
# ----------------------------------------------
# MOST COMMON WORDS AND VOC SIZRS (TODO)
# ----------------------------------------------
from nltk.corpus import stopwords
stops_nl = list(stopwords.words('dutch'))

import string

def get_most_freq(vocab, stopwords = None, k = 10):
    tmp_vocab = vocab
    if stopwords is not None:
        for word in stopwords:
            if word in tmp_vocab:
                del tmp_vocab[word]
    sorted_vocab = dict(sorted(tmp_vocab.items(), key=lambda item: item[1], reverse=True))
    
    print([w + " " + str(sorted_vocab[w]) for w in list(sorted_vocab.keys())[:k]])
    
# We also remove punctuation
print("Domain & model & voc size\\\\\hline")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end="")
    print(" & Ref.", end=" & ")
    vocab_size, _vocab  = get_vocabulary([s.translate(str.maketrans("", "", string.punctuation)) for s in refs_tok[translation_name]])
    vocab_size_low, _vocab  = get_vocabulary_lowercase([s.translate(str.maketrans("", "", string.punctuation)) for s in refs_tok[translation_name]])
    print(str(vocab_size) + "\\\\\\hline")
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        vocab_size, _vocab  = get_vocabulary([s.translate(str.maketrans("", "", string.punctuation)) for s in all_tok_translations[exp_prefix]])
        vocab_size_low, _vocab  = get_vocabulary_lowercase([s.translate(str.maketrans("", "", string.punctuation)) for s in all_tok_translations[exp_prefix]])
        print(str(vocab_size) + "\\\\\\hline")



Domain & model & voc size\\\hline
news & Ref. & 17995\\\hline
 & mbart & 17641\\\hline
 & j2 & 17849\\\hline
 & gpt & 17974\\\hline
 & deepseek & 25088\\\hline
poetry & Ref. & 6651\\\hline
 & mbart & 6815\\\hline
 & j2 & 6828\\\hline
 & gpt & 6860\\\hline
 & deepseek & 17707\\\hline
literature & Ref. & 18326\\\hline
 & mbart & 18532\\\hline
 & j2 & 18638\\\hline
 & gpt & 18909\\\hline
 & deepseek & 26290\\\hline


In [26]:
print("Most frequent words")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end = " & " )
    print("Ref.", end=" & ")
    _vocab_size_low, vocab  = get_vocabulary_lowercase([s.translate(str.maketrans("", "", string.punctuation)) for s in refs_tok[translation_name]])    
    print("Stopwords Incl.:", end=" ")
    get_most_freq(vocab)
    print("\\\\\\hline")
    print(" & & Stopwords Excl.:", end=" ")
    get_most_freq(vocab, stops_nl)
    print("\\\\\\hline")
    for model_name in ["mbart", "j2", "gpt"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        _vocab_size_low, vocab  = get_vocabulary_lowercase([s.translate(str.maketrans("", "", string.punctuation)) for s in all_tok_translations[exp_prefix]])
        print("Stopwords Incl.:", end=" ")
        get_most_freq(vocab)
        print("\\\\\\hline")
        print("& & Stopwords Excl.:", end=" ")
        get_most_freq(vocab, stops_nl)
        print("\\\\\\hline")

Most frequent words
news & Ref. & Stopwords Incl.: ['de 1114', 'van 583', 'een 536', 'het 507', 'in 413', 'en 395', 'dat 272', 'is 262', 'te 246', 'voor 229']
\\\hline
 & & Stopwords Excl.: ['we 129', 'wel 54', 'jaar 42', 's 38', 'onze 35', 'twee 32', 'tussen 30', 'leuven 30', 'onderzoek 29', 'één 28']
\\\hline
 & mbart & Stopwords Incl.: ['de 1068', 'van 603', 'een 533', 'het 520', 'en 410', 'in 407', 'is 305', 'te 249', 'zijn 240', 'dat 229']
\\\hline
& & Stopwords Excl.: ['we 144', 'mensen 54', 'onze 49', 's 46', 'jaar 46', 'leuven 39', 'onderzoek 36', 'universiteit 36', 'twee 35', 'zegt 31']
\\\hline
 & j2 & Stopwords Incl.: ['de 1052', 'van 611', 'een 545', 'het 521', 'en 433', 'in 388', 'is 282', 'te 265', 'zijn 226', 'dat 225']
\\\hline
& & Stopwords Excl.: ['we 139', 'onze 47', 'jaar 43', 'leuven 40', 'mensen 37', 'universiteit 37', 's 36', 'onderzoek 35', 'twee 35', 'moeten 31']
\\\hline
 & gpt & Stopwords Incl.: ['de 1059', 'van 633', 'een 539', 'het 524', 'en 432', 'in 379',

In [12]:
# Get Average sentence length
def get_average_sentlength(sentences):
    all_lengths = [len(s.strip().split()) for s in sentences]
    return round(sum(all_lengths) / len(all_lengths), 3)

print("Domain & model & avg. sent length\\\\\hline")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end="")
    print(" & Ref.", end=" & ")
    print(str(get_average_sentlength(refs_tok[translation_name])) + "\\\\\hline")
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        print(str(get_average_sentlength(all_tok_translations[exp_prefix])) + "\\\\\hline")

Domain & model & avg. sent length\\\hline
news & Ref. & 20.257\\\hline
 & mbart & 19.979\\\hline
 & j2 & 19.876\\\hline
 & gpt & 19.952\\\hline
 & deepseek & 27.961\\\hline
poetry & Ref. & 7.444\\\hline
 & mbart & 8.204\\\hline
 & j2 & 8.272\\\hline
 & gpt & 7.92\\\hline
 & deepseek & 20.367\\\hline
literature & Ref. & 21.226\\\hline
 & mbart & 21.637\\\hline
 & j2 & 21.534\\\hline
 & gpt & 21.751\\\hline
 & deepseek & 30.219\\\hline


In [35]:
sent_num = 42
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name, end="")
    print(" & Ref.", end=" & ")
    print(refs_tok[translation_name][sent_num])
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        print(" & " + model_name, end=" & ")
        print(all_tok_translations[exp_prefix][sent_num])

news & Ref. & Ik heb de klas een namiddag van haar overgenomen , en toen heb ik de kinderen - ze waren tussen 8 en 16 jaar oud - gevraagd om typische township-situaties uit te beelden .
 & mbart & Ik vervangde haar op een middag in de klas en vroeg de kinderen - tussen de acht en sestien jaar - om typische situaties in een stad te schilderen .
 & j2 & Ik verving haar een middag in de klas en vroeg de kinderen , in de leeftijd van acht tot zestien jaar , om typische townshipsituaties te schetsen .
 & gpt & Ik verving haar op een middag in de klas , en ik vroeg de kinderen - tussen acht en zestien jaar oud - om typische situaties in de township uit te beelden .
 & deepseek & Op een middag verving ik haar tijdens de les en vroeg ik de kinderen - van acht tot zestien jaar oud - om typische situaties in de township te beschrijven .
poetry & Ref. & een met wit vlies bedekte ogen
 & mbart & De ogen bedekt met witte film .
 & j2 & ogen bedekt met een witte film .
 & gpt & ogen bedekt met een w

# Regression

In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 46.1 MB/s  0:00:00 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]━━━━ 1/2 [gensim]


In [27]:
import logging
import pandas as pd
import numpy as np
from numpy import random
import gensim
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import re

%matplotlib inline
nltk.download('stopwords')

# df = pd.read_csv('combined-lit.csv', encoding = 'ISO-8859-2',sep = ";", header = 0)
# df['translation'] = df['translation'].fillna("")
# df = df[pd.notnull(df['model'])]
# print(df.head(10))
# print(df['translation'].apply(lambda x: len(x.split(' '))).sum())

REPLACE_BY_SPACE_RE = re.compile('[/(){}\[\]\|@,;]')
BAD_SYMBOLS_RE = re.compile('[^0-9a-z #+_]')
STOPWORDS = set(stopwords.words('dutch'))

def clean_text(text):
    """
        text: a string
        
        return: modified initial string
    """
    text = text.lower() # lowercase text
    text = REPLACE_BY_SPACE_RE.sub(' ', text) # replace REPLACE_BY_SPACE_RE symbols by space in text
    text = BAD_SYMBOLS_RE.sub('', text) # delete symbols which are in BAD_SYMBOLS_RE from text
    text = ' '.join(word for word in text.split() if word not in STOPWORDS) # delete stopwors from text
    return text

combined_X = {} # the key is the domain the rest are just sentences
combined_y = {} # the key is the domain the rest are just the model names
combined_X_norefs = {} # the key is the domain the rest are just sentences
combined_y_norefs = {} # the key is the domain the rest are just the model names

for translation_name in ["news", "poetry", "literature"]:
    print(translation_name)
    combined_X[translation_name] = [clean_text(sent) for sent in refs_tok[translation_name]]
    combined_y[translation_name] = ["ref" for _ in range(len(refs_tok[translation_name]))]    
    combined_X_norefs[translation_name] = []
    combined_y_norefs[translation_name] = []
    for model_name in ["mbart", "j2", "gpt", "deepseek"]:
        exp_prefix = model_name+"_"+translation_name
        combined_X[translation_name] = combined_X[translation_name] + [clean_text(sent) for sent in all_tok_translations[exp_prefix]]
        combined_y[translation_name] = combined_y[translation_name] + [model_name for _ in range(len(all_tok_translations[exp_prefix]))]    
        combined_X_norefs[translation_name] = combined_X_norefs[translation_name] + [clean_text(sent) for sent in all_tok_translations[exp_prefix]]
        combined_y_norefs[translation_name] = combined_y_norefs[translation_name] + [model_name for _ in range(len(all_tok_translations[exp_prefix]))]    

# def print_plot(index):
#     example = df[df.index == index][['translation', 'model']].values[0]
#     if len(example) > 0:
#         print(example[0])
#         print('Tag:', example[1])
# print_plot(10)

news
poetry
literature


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dimitarsh1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import GridSearchCV

def get_logreg_best(X_train, X_test, y_train, y_test):
    logreg = Pipeline([('vect', CountVectorizer()),
                    ('tfidf', TfidfTransformer()),
                    ('lr', LogisticRegression(n_jobs=-1, C=1e5)),
                   ])
    
    grid_params = {
      'lr__penalty': ['l1', 'l2'],
      'lr__C': [1, 10, 100, 1000],
      'lr__max_iter': [20, 50, 100, 150, 200],
      'lr__solver': ['newton-cg', 'lbfgs', 'sag', 'saga']
    }
    clf = GridSearchCV(logreg, grid_params)
    clf.fit(X_train, y_train)
    print("Best Score: ", clf.best_score_)
    print("Best Params: ", clf.best_params_)


In [84]:
for translation_name in ["news", "poetry", "literature"]:
    X_train, X_test, y_train, y_test = train_test_split(combined_X[translation_name], combined_y[translation_name], test_size=0.3, random_state = 42)
    get_logreg_best(X_train, X_test, y_train, y_test)


/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/sit

Best Score:  0.2250077693576534
Best Params:  {'lr__C': 1, 'lr__max_iter': 50, 'lr__penalty': 'l1', 'lr__solver': 'saga'}


/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/sit

Best Score:  0.20342857142857143
Best Params:  {'lr__C': 1, 'lr__max_iter': 20, 'lr__penalty': 'l1', 'lr__solver': 'saga'}


/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/dimitarsh1/anaconda3/lib/python3.13/sit

Best Score:  0.21657142857142858
Best Params:  {'lr__C': 1, 'lr__max_iter': 20, 'lr__penalty': 'l1', 'lr__solver': 'saga'}


/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
300 fits failed out of a total of 800.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/dimitarsh1/anaconda3/lib/python3.13/site-packages/sklearn/pipel

In [87]:
def get_logreg(X_train, X_test, y_train, y_test):
    logreg = Pipeline([('vect', CountVectorizer()),
                    ('tfidf', TfidfTransformer()),
                    ('lr', LogisticRegression(n_jobs=1, C=1, max_iter=150, penalty='l1', solver='saga')),
                   ])
    
    logreg.fit(X_train, y_train)
    
    y_pred = logreg.predict(X_test)
    
    print('accuracy %s' % accuracy_score(y_pred, y_test))
    print(classification_report(y_test, y_pred,target_names=set(y_train+y_test)))

In [88]:
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name)
    X_train, X_test, y_train, y_test = train_test_split(combined_X_norefs[translation_name], combined_y_norefs[translation_name], test_size=0.3, random_state = 42)
    get_logreg(X_train, X_test, y_train, y_test)

print("With reference")
for translation_name in ["news", "poetry", "literature"]:
    print(translation_name)
    X_train, X_test, y_train, y_test = train_test_split(combined_X[translation_name], combined_y[translation_name], test_size=0.3, random_state = 42)
    get_logreg(X_train, X_test, y_train, y_test)

news
accuracy 0.2518518518518518
              precision    recall  f1-score   support

         gpt       0.39      0.22      0.28       288
    deepseek       0.23      0.63      0.34       299
          j2       0.23      0.09      0.13       310
       mbart       0.23      0.09      0.13       318

    accuracy                           0.25      1215
   macro avg       0.27      0.26      0.22      1215
weighted avg       0.27      0.25      0.22      1215

poetry
accuracy 0.24
              precision    recall  f1-score   support

         gpt       0.41      0.16      0.23       284
    deepseek       0.23      0.69      0.35       294
          j2       0.19      0.05      0.08       307
       mbart       0.17      0.07      0.10       315

    accuracy                           0.24      1200
   macro avg       0.25      0.24      0.19      1200
weighted avg       0.25      0.24      0.19      1200

literature
accuracy 0.25333333333333335
              precision    recall  f

In [64]:
import numpy as np

from sklearn import metrics
from sklearn.cluster import DBSCAN

for i in range(1, 20):
    print(i)
    dbs = Pipeline([('vect', CountVectorizer()),
                        ('tfidf', TfidfTransformer()),
                       ('db', DBSCAN(eps=0.3, min_samples=i)),
                    ])
    
    for translation_name in ["news", "poetry", "literature"]:
        print(translation_name)
    
        dbs.fit(combined_X_norefs[translation_name])
        labels = dbs.named_steps['db'].labels_
        
        # Number of clusters in labels, ignoring noise if present.
        n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise_ = list(labels).count(-1)
        
        print("Estimated number of clusters: %d" % n_clusters_)
        print("Estimated number of noise points: %d" % n_noise_)
    
    print("With refernce")
    for translation_name in ["news", "poetry", "literature"]:
        print(translation_name)
        dbs.fit(combined_X[translation_name])
        labels = dbs.named_steps['db'].labels_
        
        # Number of clusters in labels, ignoring noise if present.
        n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise_ = list(labels).count(-1)
        
        print("Estimated number of clusters: %d" % n_clusters_)
        print("Estimated number of noise points: %d" % n_noise_)

1
news
Estimated number of clusters: 3438
Estimated number of noise points: 0
poetry
Estimated number of clusters: 3008
Estimated number of noise points: 0
literature
Estimated number of clusters: 3402
Estimated number of noise points: 0
With refernce
news
Estimated number of clusters: 4342
Estimated number of noise points: 0
poetry
Estimated number of clusters: 3756
Estimated number of noise points: 0
literature
Estimated number of clusters: 4279
Estimated number of noise points: 0
2
news
Estimated number of clusters: 432
Estimated number of noise points: 3006
poetry
Estimated number of clusters: 615
Estimated number of noise points: 2393
literature
Estimated number of clusters: 403
Estimated number of noise points: 2999
With refernce
news
Estimated number of clusters: 474
Estimated number of noise points: 3868
poetry
Estimated number of clusters: 683
Estimated number of noise points: 3073
literature
Estimated number of clusters: 444
Estimated number of noise points: 3835
3
news
Estim